# Phase 13 — Live local LLM smoke test
One executable cell. Start the **local** FastAPI process and install the optional LLM dependencies first. No prior benchmarks are rerun. This tests only ungrounded `/generate`, not RAG.


In [ ]:
# PHASE 13 - one-cell LIVE smoke test. Run the local FastAPI server first.
# Local Jupyter recommended: Colab localhost cannot reach your Windows server.
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

BASE_URL = "http://127.0.0.1:8000"
with urlopen(BASE_URL + "/health", timeout=15) as reply:
    health = json.load(reply)
assert health["status"] == "ok", health

payload = {
    "prompt": "Γράψε μια σύντομη πρόταση στα ελληνικά για την ενημέρωση των πολιτών.",
    "max_new_tokens": 24,
}
request = Request(
    BASE_URL + "/generate",
    data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urlopen(request, timeout=900) as reply:
        status = reply.status
        result = json.load(reply)
except HTTPError as exc:
    print("HTTP", exc.code, exc.read().decode("utf-8", errors="replace"))
    raise
assert status == 200 and result["mode"] == "local_llm", result
assert result["model"] and result["device"] in ("cpu", "cuda")
assert 0 <= result["output_tokens"] <= payload["max_new_tokens"]
print("PHASE 13 LIVE GENERATION SMOKE PASSED")
print(json.dumps(result, indent=2, ensure_ascii=False))
print("NOTE: this validates model execution, not factuality or RAG quality.")
